# 🎯 Model Training with Optuna Hyperparameter Optimization and unbalanced data handling

This notebook implements model training with balanced classes using Optuna for hyperparameter optimization. The process includes:

- 📊 Data loading and preprocessing\n",
- 🔄 Cross-validation with stratified k-folds\n",
- ⚖️ Class balancing using RandomUnderSampler\n",
- 🎮 Hyperparameter optimization with Optuna\n",
- 📈 Model evaluation and visualization\n",
- 💾 Model persistence\n",

## Table of Contents
1. [Import Libraries and Load Data](#import-libraries),
2. [Support Functions](#support-functions),
3. [Hyperparameter Optimization](#optimization),
4. [Visualization of Results](#visualization),
5. [Final Model Training](#final-model),
6. [Model Evaluation](#evaluation)


In [1]:
%reload_ext autoreload
%autoreload 2

## 📚 Import Libraries and Load Data <a name="import-libraries"></a>

In [2]:
# Standard library imports
import pandas as pd
import numpy as np
import sys
from functools import partial
from statistics import median
from tqdm import tqdm

# Machine Learning imports
from lightgbm import LGBMClassifier
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, precision_recall_curve

# Optimization library
import optuna

# Custom utilities
sys.path.append('../../')
from utils.custom_kfolds import CustomStratifiedKFold

# Load training data
X = pd.read_parquet('../../data/train_data.parquet')
metadata_columns = ['trans_date_trans_time', 'gender', 'street']
train_y = X['is_fraud']
train_X = X.drop(columns=['is_fraud'] + metadata_columns)

# Load holdout and OOT data
holdout = pd.read_parquet('../../data/holdout_data.parquet')
holdout_y = holdout['is_fraud']
holdout_X = holdout.drop(columns=['is_fraud'] + metadata_columns)

oot = pd.read_parquet('../../data/oot_data.parquet')
oot_y = oot['is_fraud']
oot_X = oot.drop(columns=['is_fraud'] + metadata_columns)

## 🛠️ Support Functions <a name="support-functions"></a>

In [4]:
def model_training_kfold(train_X, train_y, hyperparameters, nr_folds=3):
    """Train a model using k-fold cross validation with undersampling.
    
    Args:
        train_X (pd.DataFrame): Training features
        train_y (pd.Series): Training labels
        hyperparameters (dict): Model hyperparameters
        nr_folds (int): Number of folds for cross-validation
        
    Returns:
        tuple: (validation_scores, training_scores, trained_model)
    """
    # Initialize cross-validation and undersampling
    skf = StratifiedKFold(n_splits=nr_folds,shuffle=True,random_state=42)
    undersample_func=RandomUnderSampler(sampling_strategy=0.2, random_state=42)
    # initialize the model
    lgbm_model = LGBMClassifier( random_state=42, n_jobs=-1, metric="average_precision" )

    train_scores: list[float] = []
    val_scores: list[float] = []

    for _, (train_index, test_index)  in tqdm(enumerate(skf.split(train_X, train_y)), desc="Generating K-Folds", total=3):
        # Split data
        train_df, y_train = train_X.iloc[train_index], train_y.iloc[train_index]
        test_df, y_test = train_X.iloc[test_index], train_y.iloc[test_index]

        # Undersample the training data
        train_df, y_train = undersample_func.fit_resample(train_df, y_train)

        # Fit the classifier on the training data
        lgbm_model.set_params(**hyperparameters)

        # Fit the classifier on the training data   
        lgbm_model.fit(
            train_df,
            y_train,
            eval_set=[(test_df, y_test)]
        )        

        # Make predictions
        y_pred = lgbm_model.predict(test_df)#[:,1]
        y_pred_train = lgbm_model.predict(train_X)#[:,1]

        # Calculate precision scores
        train_scores.append(precision_score(train_y, y_pred_train))
        val_scores.append(precision_score(y_test, y_pred))     

    return val_scores, train_scores, lgbm_model



def objective(trial: optuna.Trial, nr_folds: int = 3):
    """Optuna objective function for hyperparameter optimization.
    
    Args:
        trial (optuna.Trial): Optuna trial object
        nr_folds (int): Number of folds for cross-validation
        
    Returns:
        float: Median validation score across folds
    """
    # Define hyperparameters
    hyperparameters = {
        "objective": "binary",
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "early_stopping_round": 10,
        "first_metric_only": True,
        "num_leaves": trial.suggest_int("num_leaves", 2, (2**6) - 1, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 2, 256),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.5,log=True),
        'reg_alpha' : trial.suggest_int('reg_alpha',1,6),
        #'reg_lambda' : trial.suggest_int('reg_lambda',1,4),
        'n_jobs' : -1
    }

    val_scores, _, _ = model_training_kfold(train_X, train_y, hyperparameters, nr_folds=nr_folds)

    return median(val_scores)
    


## 🎮 Hyperparameter Optimization <a name="optimization"></a>

In [ ]:
print("🔍 Starting Optuna Hyperparameter Optimization")
print("Optuna Tuning")
#hyperparameters, best_score, 
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
objective = partial(objective, nr_folds=5)
study.optimize(objective, n_trials=30)

[I 2025-03-27 15:30:52,708] A new study created in memory with name: no-name-9d73c94a-4094-4038-a98b-59c26b434c6a


🔍 Starting Optuna Hyperparameter Optimization
Optuna Tuning


Generating K-Folds:   0%|          | 0/3 [00:00<?, ?it/s]

[LightGBM] [Warning] min_data_in_leaf is set=129, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=129
[LightGBM] [Warning] min_data_in_leaf is set=129, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=129
[LightGBM] [Info] Number of positive: 4888, number of negative: 24440
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000593 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2123
[LightGBM] [Info] Number of data points in the train set: 29328, number of used features: 14
[LightGBM] [Warning] min_data_in_leaf is set=129, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166667 -> initscore=-1.609438
[LightGBM] [Info] Start training from score -1.609438
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[63]	valid_0's average_precision: 0.851

Generating K-Folds:  33%|███▎      | 1/3 [00:01<00:03,  1.52s/it]

...

[LightGBM] [Warning] min_data_in_leaf is set=17, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=17
[LightGBM] [Warning] min_data_in_leaf is set=17, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=17
[LightGBM] [Info] Number of positive: 4889, number of negative: 24445
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000511 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2118
[LightGBM] [Info] Number of data points in the train set: 29334, number of used features: 14
[LightGBM] [Warning] min_data_in_leaf is set=17, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166667 -> initscore=-1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 10 rounds
[Ligh

/Users/valeria.verzi/PlayroomCode/Fraud-Modeling/fraud-modeling/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/valeria.verzi/PlayroomCode/Fraud-Modeling/fraud-modeling/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Generating K-Folds: 4it [00:02,  1.31it/s]                       

[LightGBM] [Warning] min_data_in_leaf is set=17, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=17
[LightGBM] [Warning] min_data_in_leaf is set=17, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=17
[LightGBM] [Info] Number of positive: 4889, number of negative: 24445
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000484 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2122
[LightGBM] [Info] Number of data points in the train set: 29334, number of used features: 14
[LightGBM] [Warning] min_data_in_leaf is set=17, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166667 -> initscore=-1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 10 rounds
[Ligh

/Users/valeria.verzi/PlayroomCode/Fraud-Modeling/fraud-modeling/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/valeria.verzi/PlayroomCode/Fraud-Modeling/fraud-modeling/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Generating K-Folds: 5it [00:03,  1.40it/s]
[I 2025-03-27 15:31:21,504] Trial 7 finished with value: 0.0 and parameters: {'n_estimators': 226, 'num_leaves': 47, 'min_data_in_leaf': 17, 'learning_rate': 0.007707790754868581, 'reg_alpha': 6}. Best is trial 2 with value: 0.358093

[LightGBM] [Warning] min_data_in_leaf is set=29, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=29
[LightGBM] [Warning] min_data_in_leaf is set=29, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=29
[LightGBM] [Info] Number of positive: 4888, number of negative: 24440
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000556 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2123
[LightGBM] [Info] Number of data points in the train set: 29328, number of used features: 14
[LightGBM] [Warning] min_data_in_leaf is set=29, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166667 -> initscore=-1.609438
[LightGBM] [Info] Start training from score -1.609438
Training until validation scores don't improve for 10 rounds


In [ ]:
print('📊 Number of finished trials:', len(study.trials))

In [ ]:
print('🏆 Best parameters found:')
study.best_trial.params

In [ ]:
print('📈 Best trial report:')
study.best_trial.report

## 📊 Visualization of Results <a name="visualization"></a>

In [ ]:
print('📈 Optimization History')
optuna.visualization.plot_optimization_history(study)

In [ ]:
print('🎯 Parameter Importances')
optuna.visualization.plot_param_importances(study)

In [ ]:
print('🔄 Parallel Coordinate Plot')
optuna.visualization.plot_parallel_coordinate(study)

In [ ]:
print('📊 Empirical Distribution Function')
optuna.visualization.plot_edf(study)

## 🎯 Final Model Training <a name="final-model"></a>

In [155]:
best_params = study.best_trial.params

### 📊 Model Stability Across Folds

In [ ]:
nr_folds = 10
val_scores, train_scores, lgbm_model = model_training_kfold(train_X, train_y, best_params, nr_folds=nr_folds)

In [ ]:
from matplotlib import pyplot as plt
folds = range(1, nr_folds+1)
plt.plot(folds, train_scores, 'o-', color='green', label='train')
plt.plot(folds, val_scores, 'o-', color='red', label='test')
plt.legend()
plt.grid()
plt.xlabel('Number of fold')
plt.ylabel('avg precision')
plt.show()

## Build final model

In [ ]:

model = LGBMClassifier(**best_params)
undersample_func=RandomUnderSampler(sampling_strategy=0.2, random_state=42)
train_df, y_train = undersample_func.fit_resample(train_X, train_y)
model.fit(train_df, y_train)

# Evaluate the model
y_pred = model.predict(holdout_X)
y_pred_proba = model.predict_proba(holdout_X)[:, 1]

## 💾 Save Final Model

In [159]:
import pickle
print('💾 Saving model to disk...')
with open('../../models/optuna_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('✅ Model saved successfully!')

## 📊 Model Evaluation <a name="evaluation"></a>

In [160]:
import sys
sys.path.append('../../')
from utils.eval_plots import EvalPlots
from utils.model_performance_report import ModelPerformanceReport

report_class = Report()
eval_plots_class = EvalPlots()

In [161]:
# Load evaluation data
holdout = pd.read_parquet('../../data/holdout_data.parquet')
oot = pd.read_parquet('../../data/oot_data.parquet')

In [162]:
# Prepare evaluation features
holdout_X = holdout.drop(columns=['is_fraud']+metadata_columns)[train_X.columns]
holdout_y = holdout['is_fraud']

oot_X = oot.drop(columns=['is_fraud']+metadata_columns)[train_X.columns]
oot_y = oot['is_fraud']

In [163]:
# Initialize evaluation classes
report_class = ModelPerformanceReport(train_X,train_y,holdout_X,holdout_y,oot_X,oot_y)
eval_plots = EvalPlots()

In [ ]:
report_class.produce_report(model)

In [ ]:
print('📈 Generating Prediction Distributions')
y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_true, y_oot_pred = report_class.predictions(model)

In [ ]:
report_class.plot_eval_pred_dist(y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred)

In [ ]:
print('🎯 Generating PR-AUC Report')
report_class.produce_pr_auc_report(model)